In [1]:
import json, numpy as np, pandas as pd

filas = [json.loads(l) for l in open("../data/nested_resultados.jsonl")]
err = [r for r in filas if "error" in r]
ok = [r for r in filas if "error" not in r]
print(len(filas), "líneas;", len(err), "con error")
if err:
    print(pd.Series([r["error"][:80] for r in err]).value_counts().head())

ncall = np.array([r[b]["ncall"] for r in ok for b in "gr"])
seg = np.array([r[b]["seg"] for r in ok for b in "gr"])
print("bandas que llegaron al tope de 150000 evaluaciones:", int((ncall >= 150000).sum()), "de", len(ncall))
print("segundos por banda: mediana", np.median(seg), "| máximo", seg.max())

4902 líneas; 0 con error
bandas que llegaron al tope de 150000 evaluaciones: 0 de 9804
segundos por banda: mediana 1.9 | máximo 8.3


In [2]:
NOM = ["A", "beta", "gamma", "t0", "tau_rise", "tau_fall", "sigma_extra"]
clase = pd.read_csv("../data/features.csv", usecols=["NAME", "CLASE"]).set_index("NAME")["CLASE"]

cmp_ = []
for r in ok:
    for b in "gr":
        d = r[b]
        if d["curve_fit"] is None:
            continue
        for i, p in enumerate(NOM[:6]):
            fuera = not (d["p16"][i] <= d["curve_fit"][i] <= d["p84"][i])
            cmp_.append({"CLASE": clase[r["NAME"]], "param": p, "fuera": fuera})
cmp_ = pd.DataFrame(cmp_)
print(cmp_.pivot_table(index="param", columns="CLASE", values="fuera", aggfunc="mean").round(2))

CLASE     SLSN-I  SN II  SN IIn  SN Ia  SN Ib/c
param                                          
A           0.73   0.58    0.60   0.69     0.67
beta        0.78   0.72    0.68   0.83     0.80
gamma       0.60   0.63    0.61   0.51     0.53
t0          0.68   0.59    0.55   0.62     0.63
tau_fall    0.30   0.37    0.40   0.20     0.23
tau_rise    0.49   0.59    0.47   0.33     0.43


In [3]:
filas_f = []
for r in ok:
    n = r["NAME"]
    c = pd.read_parquet(f"../data/processed/{n}.parquet")
    fmax_r = c.loc[c["banda"] == "r", "flujo"].max()
    mr = dict(zip(NOM, r["r"]["med"]))
    mg = dict(zip(NOM, r["g"]["med"]))
    fila = {"NAME": n, "CLASE": clase[n]}
    for p in ["beta", "gamma", "tau_rise", "tau_fall"]:
        fila[f"{p}_r"] = mr[p]
    fila["sigma_extra_r"] = mr["sigma_extra"] / fmax_r      
    for p in ["A", "beta", "gamma", "tau_rise", "tau_fall", "sigma_extra"]:
        fila[f"{p}_g"] = np.log10(mg[p] / mr[p])
    fila["t0_g"] = mg["t0"] - mr["t0"]
    filas_f.append(fila)

Xn = pd.DataFrame(filas_f)
Xn = Xn[Xn["NAME"].isin(clase.index)] 
print(Xn.shape)
Xn.to_csv("../data/features_nested.csv", index=False)

(4902, 14)
